<a href="https://colab.research.google.com/github/Uchihatt/My-Projects/blob/main/Marketing_Mix_Model_(MMM)_for_ROI_(Multiple_Regression).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("marketing_spend_weekly.csv")
df["week_start"] = pd.to_datetime(df["week_start"])
df = df.sort_values("week_start").reset_index(drop=True)
df.head()


,week_start,spend_tv,spend_search,spend_social,spend_email,avg_discount_rate,website_leads,conversion_rate,sales_revenue
0,2024-01-01,13700.0,24946.0,19628.0,1095.0,0.0978,6385,0.0929,301316.25
1,2024-01-08,48447.0,34017.0,5138.0,4924.0,0.0989,8482,0.1000,327827.14
2,2024-01-15,40822.0,25102.0,8249.0,8311.0,0.0498,8980,0.0902,337488.34
3,2024-01-22,17310.0,28131.0,8763.0,4652.0,0.1014,6939,0.1000,303349.00
4,2024-01-29,21088.0,21518.0,17609.0,2803.0,0.1380,6802,0.0896,287864.50


In [6]:
spend_cols = ["spend_tv","spend_search","spend_social","spend_email"]

for c in spend_cols:
    df[f"log_{c}"] = np.log1p(df[c])
LAGS = [1,2,3,4]
for c in spend_cols:
    for l in LAGS:
        df[f"log_{c}_lag{l}"] = df[f"log_{c}"].shift(l)
for c in spend_cols:
    df[f"log_{c}_roll3"] = df[f"log_{c}"].rolling(3).mean()


In [4]:
# @title Seasonality

df["month"] = df["week_start"].dt.month
df["weekofyear"] = df["week_start"].dt.isocalendar().week.astype(int)

# simple cyclical encoding (nice for models)
df["sin_week"] = np.sin(2*np.pi*df["weekofyear"]/52)
df["cos_week"] = np.cos(2*np.pi*df["weekofyear"]/52)


In [5]:
df_model = df.dropna().copy()
df_model.shape


(104, 17)

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("marketing_spend_weekly.csv")
df["week_start"] = pd.to_datetime(df["week_start"])
df = df.sort_values("week_start").reset_index(drop=True)

spend_cols = ["spend_tv","spend_search","spend_social","spend_email"]

# 1) log features
for c in spend_cols:
    df[f"log_{c}"] = np.log1p(df[c])

# 2) lag features (create lag1 and lag2)
for c in spend_cols:
    for l in [1,2]:
        df[f"log_{c}_lag{l}"] = df[f"log_{c}"].shift(l)

# 3) seasonality controls
df["weekofyear"] = df["week_start"].dt.isocalendar().week.astype(int)
df["sin_week"] = np.sin(2*np.pi*df["weekofyear"]/52)
df["cos_week"] = np.cos(2*np.pi*df["weekofyear"]/52)

# 4) drop rows that became NaN due to lags
df_model = df.dropna().copy()

# confirm lag columns exist
[c for c in df_model.columns if "lag" in c][:20]
target = "sales_revenue"

feature_cols = (
    [f"log_{c}" for c in spend_cols] +
    [f"log_{c}_lag{l}" for c in spend_cols for l in [1,2]] +
    ["avg_discount_rate", "sin_week", "cos_week"]
)

X = df_model[feature_cols]
y = df_model[target]

X.shape, y.shape


((102, 15), (102,))

In [10]:
# @title Train/Test Split
split_idx = int(len(df_model)*0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


In [11]:
# @title Scaling + Models
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, r2_score

def fit_eval(model, name):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(name,
          "MAE:", round(mean_absolute_error(y_test, pred),2),
          "R2:", round(r2_score(y_test, pred),3))
    return pipe

ols = fit_eval(LinearRegression(), "OLS")
ridge = fit_eval(Ridge(alpha=1.0), "Ridge")
lasso = fit_eval(Lasso(alpha=0.02), "Lasso")



OLS MAE: 27112.76 R2: -0.705
Ridge MAE: 26756.04 R2: -0.671
Lasso MAE: 27112.7 R2: -0.705


In [12]:
#Multi Colinearity Check
!pip -q install statsmodels
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = sm.add_constant(X_train)
vif = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif.sort_values("VIF", ascending=False).head(15)


,feature,VIF
0,const,4182.778378
6,log_spend_tv_lag2,1.361943
13,avg_discount_rate,1.299104
11,log_spend_email_lag1,1.292759
7,log_spend_search_lag1,1.282456
2,log_spend_search,1.255225
12,log_spend_email_lag2,1.251926
8,log_spend_search_lag2,1.236979
9,log_spend_social_lag1,1.224513
4,log_spend_email,1.211325


In [13]:
#Contribution by Channel (Model Based)
coef = ridge.named_steps["model"].coef_
cols = X_train.columns
coef_df = pd.DataFrame({"feature": cols, "coef": coef}).sort_values("coef", ascending=False)
coef_df.head(20)


,feature,coef
13,sin_week,19681.491596
1,log_spend_search,12168.453895
6,log_spend_search_lag1,6160.878563
8,log_spend_social_lag1,6094.407887
3,log_spend_email,6029.670221
2,log_spend_social,4371.894210
9,log_spend_social_lag2,4032.469018
0,log_spend_tv,3086.169581
7,log_spend_search_lag2,2867.672676
4,log_spend_tv_lag1,2578.097471


In [14]:
scaler = ridge.named_steps["scaler"]
X_scaled = scaler.transform(X)  # all rows
contrib = pd.DataFrame(X_scaled, columns=X.columns, index=df_model.index).mul(coef, axis=1)

contrib["base_intercept"] = ridge.named_steps["model"].intercept_
contrib["predicted_sales"] = contrib.sum(axis=1) + contrib["base_intercept"]
contrib.head()


,log_spend_tv,log_spend_search,log_spend_social,log_spend_email,log_spend_tv_lag1,log_spend_tv_lag2,log_spend_search_lag1,log_spend_search_lag2,log_spend_social_lag1,log_spend_social_lag2,log_spend_email_lag1,log_spend_email_lag2,avg_discount_rate,sin_week,cos_week,base_intercept,predicted_sales
2,2532.422696,6627.258042,-1646.680862,8937.438779,3009.308149,303.766435,6362.407157,1434.190364,-7202.040985,4658.671292,1870.243150,6505.014568,6168.054022,5133.736332,-11520.646863,305556.80321,644286.748696
3,-3265.884530,8957.975239,-1181.095480,4283.206775,2049.932830,-229.290110,3183.540549,2941.890047,-2111.534837,-4804.593238,3548.335743,-4359.775156,-550.386140,8336.309063,-10965.493205,305556.80321,616946.743970
4,-1931.654552,3475.498828,4194.424316,220.375017,-2756.335876,-157.016255,4375.165158,1464.495363,-1461.565093,-1462.070272,1688.087356,-8143.985149,-5315.791371,11341.794565,-10265.705146,305556.80321,606379.323309
5,-326.321605,5740.322899,3336.969328,-1128.569889,-1650.380717,205.060571,1572.142202,2018.309058,6042.805444,-1035.287801,64.215530,-3949.001770,-1227.438249,14106.366091,-9431.487173,305556.80321,625451.310339
6,3394.693896,-21500.996790,-2324.604749,1869.577829,-319.705883,121.744227,2730.077665,715.589653,4845.775167,3892.224343,-474.943795,-287.063200,1610.972517,16589.709964,-8475.004063,305556.80321,613501.653202


In [15]:
BASE = df_model.iloc[-1:].copy()

def predict_row(row_df):
    Xr = row_df[feature_cols]
    return ridge.predict(Xr)[0]

base_pred = predict_row(BASE)

delta = 10000  # PKR 10k
scenario = BASE.copy()
scenario["spend_search"] = scenario["spend_search"] + delta

# recompute log features affected
scenario["log_spend_search"] = np.log1p(scenario["spend_search"])

# IMPORTANT: lags won't change in this simple “next-week” demo.
# For a more realistic sim, you'd apply delta to future rows and shift.

scn_pred = predict_row(scenario)

uplift = scn_pred - base_pred
roi = uplift / delta

uplift, roi


(np.float64(8280.84778379649), np.float64(0.8280847783796489))

In [17]:
#+PKR 10,000 to Search → predicted revenue uplift ≈ 8,280.85
#→ ROI ≈ 0.828 revenue per 1 PKR

Shift PKR 50,000 from TV → Search → predicted revenue uplift ≈ 12,638.76
shift = 50000

realloc = BASE.copy()
realloc["spend_tv"] = np.maximum(realloc["spend_tv"] - shift, 0)
realloc["spend_search"] = realloc["spend_search"] + shift

realloc["log_spend_tv"] = np.log1p(realloc["spend_tv"])
realloc["log_spend_search"] = np.log1p(realloc["spend_search"])

uplift_realloc = predict_row(realloc) - base_pred
uplift_realloc


np.float64(12638.7594095752)

In [ ]:
#NOTES
#Executive Summary (copy/paste)

Objective: Estimate revenue contribution of each marketing channel and recommend budget allocation for maximum revenue.

Method: Built a weekly Marketing Mix Model using log-transformed spends (diminishing returns), lag features (carryover), seasonality controls, and Ridge regression to reduce multicollinearity risk.

Key finding: At current spending levels, Search shows stronger marginal returns than TV.

Scenario insights:

Increasing Search spend by PKR 10,000 is associated with ~PKR 8,281 incremental weekly revenue (marginal ROI ≈ 0.83x).

Reallocating PKR 50,000 from TV to Search (budget neutral) is associated with ~PKR 12,639 incremental weekly revenue.

Recommendation: Gradually shift budget from TV into Search and validate via controlled tests (geo split or incrementality) to confirm causality.

4) Add 2–3 supporting charts (makes it portfolio-ready)

These are the simplest charts that impress:

A) Actual vs Predicted (holdout)

Shows model is usable.

“We predict revenue reasonably well on holdout weeks.”

B) Marginal ROI bar chart by channel

Run the +10k simulation for each channel and plot ROIs:

Search, Social, Email, TV

Recommendation becomes visual.

C) Spend vs ROI curve (for 1 channel)

Demonstrates diminishing returns (great for MMM).

5) What to write in your “Recommendations” section

Don’t just say “shift TV → Search”. Add guardrails.

Recommendation format:

Shift 5–10% of TV budget into Search over next 4 weeks

Monitor:

blended CAC/CPA (if available)

revenue lift vs baseline weeks

If uplift holds, continue shifting until ROI equalizes (optimal point)

Why this is smart: MMM gives direction, but execution should be incremental.

6) One important improvement (so your scenarios are more “real MMM”)

Right now your note says lags don’t change in a next-week demo — correct.

To write better:

“For a realistic scenario, we simulate spend changes across future weeks and recompute lag features accordingly.”